In [ ]:
import torch
import pandas as pd

from dlem.dlem_genome import dlem_notebook

cooler_file = '/Users/tina/LoopExtrusion/data/H1.mcool'
output_path = f'/Users/tina/LoopExtrusion/results/whole_genome_test.tsv'
processed_output_path = f'/Users/tina/LoopExtrusion/results/whole_genome_test_processed.tsv'
resolution = 10_000
window_size = 200
stride = 50
dev_name ='cpu'

In [ ]:
def process_dataframe(df_result):
    """
    Process the dataframe to create wide format with weighted means for multiple chromosomes.
    
    Parameters:
    df_result: The input dataframe with columns chrom, start, end, patch_i, max_corr, 
               perc_nan, importance_weights, left, right
    
    Returns:
    DataFrame: Processed dataframe with weighted averages, sorted by chromosome and start position
    """
    # Process each chromosome separately
    result_dfs = []
    
    for chrom, chrom_df in df_result.groupby('chrom'):
        # Create a location identifier based on unique start/end combinations within this chromosome
        chrom_df = chrom_df.copy()  # Avoid SettingWithCopyWarning
        chrom_df['location_id'] = chrom_df.groupby(['start', 'end']).ngroup()
        
        # Create instance_id for each row within a location group
        chrom_df['instance_id'] = chrom_df.groupby('location_id').cumcount() + 1
        
        # Define columns that should have multiple instances
        varying_columns = ['patch_i', 'max_corr', 'perc_nan', 'importance_weights', 'left', 'right']
        
        # Get the maximum instance_id to determine how many repeats we have
        max_instances = chrom_df['instance_id'].max()
        
        # Pivot to create wide format for varying columns
        wide_df = pd.pivot(
            index='location_id',
            columns='instance_id',
            values=varying_columns,
            data=chrom_df
        )
        
        # Flatten the column names
        wide_df.columns = [f'{col[0]}_{col[1]}' for col in wide_df.columns]
        
        # Add the constant columns (start, end, chrom)
        constant_df = chrom_df.drop_duplicates('location_id')[['location_id', 'start', 'end', 'chrom']]
        
        # Join the constant columns with the wide format
        final_df = pd.merge(constant_df, wide_df, on='location_id')
        
        # Identify importance_weights columns
        importance_weight_cols = [col for col in final_df.columns if 'importance_weights_' in col]
        
        # Replace NaN values with 0 in all importance_weights columns
        final_df[importance_weight_cols] = final_df[importance_weight_cols].fillna(0)
        
        # Calculate sum of importance weights
        final_df['sum_importance_weights'] = final_df[importance_weight_cols].sum(axis=1)
        
        # Create weighted columns for left - dynamic based on actual number of instances
        weighted_left_cols = []
        for i in range(1, max_instances + 1):
            left_col = f'left_{i}'
            weight_col = f'importance_weights_{i}'
            weighted_col = f'weighted_left_{i}'
            
            # Only process if the columns exist
            if left_col in final_df.columns and weight_col in final_df.columns:
                weighted_left_cols.append(weighted_col)
                
                # If left is NaN, set weighted value to 0
                final_df[weighted_col] = np.where(
                    final_df[left_col].isna(),
                    0,
                    final_df[left_col] * final_df[weight_col]
                )
        
        # Create weighted columns for right - dynamic based on actual number of instances
        weighted_right_cols = []
        for i in range(1, max_instances + 1):
            right_col = f'right_{i}'
            weight_col = f'importance_weights_{i}'
            weighted_col = f'weighted_right_{i}'
            
            # Only process if the columns exist
            if right_col in final_df.columns and weight_col in final_df.columns:
                weighted_right_cols.append(weighted_col)
                
                # If right is NaN, set weighted value to 0
                final_df[weighted_col] = np.where(
                    final_df[right_col].isna(),
                    0,
                    final_df[right_col] * final_df[weight_col]
                )
        
        # Calculate sum of weighted values
        final_df['weighted_left_sum'] = final_df[weighted_left_cols].sum(axis=1)
        final_df['weighted_right_sum'] = final_df[weighted_right_cols].sum(axis=1)
        
        # Calculate weighted averages
        final_df['weighted_avg_left'] = final_df.apply(
            lambda row: row['weighted_left_sum'] / row['sum_importance_weights'] 
                       if row['sum_importance_weights'] > 0 else 0, 
            axis=1
        )
        
        final_df['weighted_avg_right'] = final_df.apply(
            lambda row: row['weighted_right_sum'] / row['sum_importance_weights'] 
                       if row['sum_importance_weights'] > 0 else 0, 
            axis=1
        )
        
        # Keep only the necessary columns and sort by start position
        final_df = final_df.sort_values('start')
        final_df = final_df[['start', 'end', 'chrom', 'weighted_avg_left', 'weighted_avg_right']]
        final_df.rename(columns={
            'weighted_avg_left': 'left', 
            'weighted_avg_right': 'right'
        }, inplace=True)
        
        # Add to our result list
        result_dfs.append(final_df)
    
    # Return each chromosome's data as a separate DataFrame in a dictionary
    # This avoids the index collision issue with 'start' values
    if len(result_dfs) == 0:
        return pd.DataFrame()  # Return empty DataFrame if no results
    elif len(result_dfs) == 1:
        # If only one chromosome, set index and return directly
        result = result_dfs[0]
        result = result.set_index('start')
        return result
    else:
        # Multiple chromosomes - create a multi-index with chrom and start
        final_df = pd.concat(result_dfs)
        final_df = final_df.set_index(['chrom', 'start'])
        return final_df

# Example usage:
# final_processed_df = process_dataframe(df_result)
# print(final_processed_df.head())

In [ ]:
# if you add chrom_subset=chrom_list, this can be also used for a subset of chroms.

df_result = dlem_notebook(cooler_file, output_path, resolution, 
                          model_name="minimal_dlem", window_size=window_size, stride=stride, perc_nan_threshold=0.3, lr=0.5, 
                          reader_name='datareader_cooler', dev_name=dev_name, do_return_result=True)

final_df = process_dataframe(df_result)
final_df_reset_index = final_df.reset_index()

final_df_reset_index.to_csv(processed_output_path , sep='\t')